# Аналитика фундаментальных данных

Ноутбук демонстрирует динамику ключевых финансовых показателей компаний:  
EBITDA, выручка, чистая прибыль, FCF, долговая нагрузка и другие метрики.

**Структура данных:**
```
data/{Сектор}/{ТИКЕР}/
    overview.csv
    income_statement_annual.csv
    income_statement_quarterly.csv
    balance_sheet_annual.csv
    balance_sheet_quarterly.csv
    cash_flow_annual.csv
    cash_flow_quarterly.csv
    earnings_annual.csv
    earnings_quarterly.csv
```

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

DATA_DIR = Path('data')
print('Директория данных:', DATA_DIR.resolve())

## 1. Вспомогательные функции

In [ ]:
def load_ticker_data(ticker: str, report_type: str) -> pd.DataFrame:
    """
    Загружает CSV-файл для заданного тикера и типа отчёта.
    Автоматически ищет тикер во всех секторах.
    """
    for sector_dir in DATA_DIR.iterdir():
        if not sector_dir.is_dir():
            continue
        path = sector_dir / ticker / f"{report_type}.csv"
        if path.exists():
            df = pd.read_csv(path)
            if 'fiscalDateEnding' in df.columns:
                df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])
                df.sort_values('fiscalDateEnding', inplace=True)
                df.reset_index(drop=True, inplace=True)
            return df
    return pd.DataFrame()


def get_sector(ticker: str) -> str:
    """Возвращает сектор тикера."""
    mapping_path = DATA_DIR / 'ticker_sector_mapping.csv'
    if mapping_path.exists():
        df = pd.read_csv(mapping_path)
        row = df[df['ticker'] == ticker]
        if not row.empty:
            return row.iloc[0]['sector']
    return 'Unknown'


def load_all_tickers_report(report_type: str, numeric_col: str,
                             sector_filter: str | None = None) -> pd.DataFrame:
    """
    Загружает один числовой столбец из заданного типа отчёта
    для всех (или отфильтрованных по сектору) тикеров.
    Возвращает сводный DataFrame: строки = даты, столбцы = тикеры.
    """
    frames = {}
    sectors = [sector_filter] if sector_filter else [
        d.name for d in DATA_DIR.iterdir() if d.is_dir() and d.name != '__pycache__'
    ]
    for sector in sectors:
        sector_dir = DATA_DIR / sector
        if not sector_dir.exists():
            continue
        for ticker_dir in sector_dir.iterdir():
            if not ticker_dir.is_dir():
                continue
            path = ticker_dir / f"{report_type}.csv"
            if not path.exists():
                continue
            try:
                df = pd.read_csv(path)
                if numeric_col not in df.columns or 'fiscalDateEnding' not in df.columns:
                    continue
                df['fiscalDateEnding'] = pd.to_datetime(df['fiscalDateEnding'])
                df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')
                s = df.set_index('fiscalDateEnding')[numeric_col].dropna()
                if not s.empty:
                    frames[ticker_dir.name] = s
            except Exception:
                pass
    if not frames:
        return pd.DataFrame()
    return pd.DataFrame(frames).sort_index()


def billions(x, pos=None):
    """Форматирует ось Y в миллиардах USD."""
    return f'{x/1e9:.1f}B'


def millions(x, pos=None):
    return f'{x/1e6:.0f}M'


print('Функции загружены.')

## 2. Обзор загруженных данных

In [ ]:
mapping_path = DATA_DIR / 'ticker_sector_mapping.csv'

if mapping_path.exists():
    mapping = pd.read_csv(mapping_path)
    print(f'Всего тикеров в базе: {len(mapping)}')
    print(f'Секторов: {mapping["sector"].nunique()}')
    print()
    sector_counts = mapping['sector'].value_counts()
    print('Распределение по секторам:')
    print(sector_counts.to_string())
else:
    print('Маппинг не найден. Сначала запустите download_fundamentals.py')

In [ ]:
# Визуализация распределения тикеров по секторам
if mapping_path.exists():
    fig, ax = plt.subplots(figsize=(12, 6))
    sector_counts.sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_xlabel('Количество компаний')
    ax.set_title('Распределение тикеров по секторам')
    for bar in ax.patches:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                str(int(bar.get_width())), va='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 3. Динамика EBITDA — отдельные компании

In [ ]:
# Список тикеров для сравнения
TICKERS = ['AAPL', 'MSFT', 'AMZN', 'GOOG', 'META']  # измените под себя

fig, axes = plt.subplots(len(TICKERS), 1, figsize=(14, 4 * len(TICKERS)))
if len(TICKERS) == 1:
    axes = [axes]

for ax, ticker in zip(axes, TICKERS):
    df = load_ticker_data(ticker, 'income_statement_annual')
    if df.empty or 'ebitda' not in df.columns:
        ax.set_title(f'{ticker} — данные не найдены')
        continue
    df['ebitda'] = pd.to_numeric(df['ebitda'], errors='coerce')
    df_plot = df.dropna(subset=['ebitda'])
    ax.bar(df_plot['fiscalDateEnding'].dt.year, df_plot['ebitda'] / 1e9,
           color='steelblue', alpha=0.8)
    ax.set_title(f'{ticker} — EBITDA (годовой)', fontsize=13)
    ax.set_ylabel('млрд USD')
    ax.set_xlabel('Год')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))

plt.tight_layout()
plt.show()

## 4. Сравнение EBITDA нескольких компаний

In [ ]:
COMPARE_TICKERS = ['AAPL', 'MSFT', 'AMZN', 'GOOG', 'META']  # измените под себя

fig, ax = plt.subplots(figsize=(14, 7))

for ticker in COMPARE_TICKERS:
    df = load_ticker_data(ticker, 'income_statement_annual')
    if df.empty or 'ebitda' not in df.columns:
        continue
    df['ebitda'] = pd.to_numeric(df['ebitda'], errors='coerce')
    df_plot = df.dropna(subset=['ebitda']).set_index('fiscalDateEnding')
    ax.plot(df_plot.index, df_plot['ebitda'] / 1e9, marker='o', label=ticker, linewidth=2)

ax.set_title('EBITDA — сравнение компаний (годовой)', fontsize=14)
ax.set_ylabel('млрд USD')
ax.set_xlabel('Год')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Динамика выручки и чистой прибыли

In [ ]:
TICKER = 'AAPL'  # измените под себя

df = load_ticker_data(TICKER, 'income_statement_annual')

if not df.empty:
    for col in ['totalRevenue', 'grossProfit', 'operatingIncome', 'netIncome', 'ebitda']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Выручка vs Чистая прибыль
    ax = axes[0]
    ax.bar(df['fiscalDateEnding'].dt.year - 0.2, df['totalRevenue'] / 1e9,
           width=0.4, label='Выручка', color='steelblue', alpha=0.8)
    ax.bar(df['fiscalDateEnding'].dt.year + 0.2, df['netIncome'] / 1e9,
           width=0.4, label='Чистая прибыль', color='seagreen', alpha=0.8)
    ax.set_title(f'{TICKER} — Выручка и чистая прибыль')
    ax.set_ylabel('млрд USD')
    ax.legend()

    # Маржи
    ax = axes[1]
    if 'grossProfit' in df.columns and 'totalRevenue' in df.columns:
        df['gross_margin'] = df['grossProfit'] / df['totalRevenue'] * 100
        df['net_margin'] = df['netIncome'] / df['totalRevenue'] * 100
        ax.plot(df['fiscalDateEnding'].dt.year, df['gross_margin'],
                marker='o', label='Валовая маржа %', color='steelblue', linewidth=2)
        ax.plot(df['fiscalDateEnding'].dt.year, df['net_margin'],
                marker='s', label='Чистая маржа %', color='seagreen', linewidth=2)
    ax.set_title(f'{TICKER} — Рентабельность')
    ax.set_ylabel('%')
    ax.legend()

    plt.tight_layout()
    plt.show()
else:
    print(f'Данные для {TICKER} не найдены')

## 6. Квартальная прибыль на акцию (EPS) и сюрпризы

In [ ]:
TICKER = 'AAPL'  # измените под себя

df = load_ticker_data(TICKER, 'earnings_quarterly')

if not df.empty:
    for col in ['reportedEPS', 'estimatedEPS', 'surprisePercentage']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.tail(20)  # последние 20 кварталов

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # EPS факт vs прогноз
    ax = axes[0]
    x = range(len(df))
    ax.plot(x, df['reportedEPS'].values, marker='o', label='Фактический EPS', color='steelblue', linewidth=2)
    if 'estimatedEPS' in df.columns:
        ax.plot(x, df['estimatedEPS'].values, marker='s', linestyle='--',
                label='Прогнозный EPS', color='orange', linewidth=2)
    ax.set_xticks(list(x))
    ax.set_xticklabels(
        [str(d.date()) for d in df['fiscalDateEnding']],
        rotation=45, ha='right', fontsize=9
    )
    ax.set_title(f'{TICKER} — EPS квартальный (факт vs прогноз)')
    ax.set_ylabel('USD')
    ax.legend()

    # Surprise %
    ax = axes[1]
    if 'surprisePercentage' in df.columns:
        colors = ['seagreen' if v >= 0 else 'tomato' for v in df['surprisePercentage'].fillna(0)]
        ax.bar(list(x), df['surprisePercentage'].fillna(0).values, color=colors, alpha=0.8)
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_xticks(list(x))
        ax.set_xticklabels(
            [str(d.date()) for d in df['fiscalDateEnding']],
            rotation=45, ha='right', fontsize=9
        )
        ax.set_title(f'{TICKER} — Отклонение EPS от прогноза (%)')
        ax.set_ylabel('%')

    plt.tight_layout()
    plt.show()
else:
    print(f'Данные для {TICKER} не найдены')

## 7. Баланс: активы, долг, собственный капитал

In [ ]:
TICKER = 'AAPL'  # измените под себя

df = load_ticker_data(TICKER, 'balance_sheet_annual')

if not df.empty:
    for col in ['totalAssets', 'totalLiabilities', 'totalShareholderEquity',
                'shortLongTermDebtTotal', 'cashAndCashEquivalentsAtCarryingValue']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Структура баланса
    ax = axes[0]
    years = df['fiscalDateEnding'].dt.year
    ax.stackplot(years,
                 df['totalShareholderEquity'].fillna(0) / 1e9,
                 df['totalLiabilities'].fillna(0) / 1e9,
                 labels=['Собственный капитал', 'Обязательства'],
                 colors=['seagreen', 'tomato'], alpha=0.7)
    ax.set_title(f'{TICKER} — Структура баланса')
    ax.set_ylabel('млрд USD')
    ax.legend(loc='upper left')

    # Долг vs Кэш
    ax = axes[1]
    if 'shortLongTermDebtTotal' in df.columns:
        ax.plot(years, df['shortLongTermDebtTotal'].fillna(0) / 1e9,
                marker='o', label='Суммарный долг', color='tomato', linewidth=2)
    if 'cashAndCashEquivalentsAtCarryingValue' in df.columns:
        ax.plot(years, df['cashAndCashEquivalentsAtCarryingValue'].fillna(0) / 1e9,
                marker='s', label='Денежные средства', color='steelblue', linewidth=2)
    ax.set_title(f'{TICKER} — Долг vs Кэш')
    ax.set_ylabel('млрд USD')
    ax.legend()

    plt.tight_layout()
    plt.show()

## 8. Free Cash Flow (FCF = Операционный CF − CAPEX)

In [ ]:
COMPARE_TICKERS = ['AAPL', 'MSFT', 'AMZN', 'GOOG']  # измените под себя

fig, ax = plt.subplots(figsize=(14, 7))

for ticker in COMPARE_TICKERS:
    df = load_ticker_data(ticker, 'cash_flow_annual')
    if df.empty:
        continue
    for col in ['operatingCashflow', 'capitalExpenditures']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'operatingCashflow' in df.columns and 'capitalExpenditures' in df.columns:
        df['fcf'] = df['operatingCashflow'] + df['capitalExpenditures']  # capex уже отрицательный
        df_plot = df.dropna(subset=['fcf'])
        ax.plot(df_plot['fiscalDateEnding'].dt.year, df_plot['fcf'] / 1e9,
                marker='o', label=ticker, linewidth=2)

ax.set_title('Free Cash Flow (FCF) — сравнение компаний', fontsize=14)
ax.set_ylabel('млрд USD')
ax.set_xlabel('Год')
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Тепловая карта EV/EBITDA по компаниям внутри сектора

In [ ]:
SECTOR = 'Technology'  # измените под себя

sector_dir = DATA_DIR / SECTOR
records = []

if sector_dir.exists():
    for ticker_dir in sector_dir.iterdir():
        ov_path = ticker_dir / 'overview.csv'
        if not ov_path.exists():
            continue
        try:
            ov = pd.read_csv(ov_path)
            row = ov.iloc[0]
            records.append({
                'Тикер': ticker_dir.name,
                'Название': str(row.get('Name', ''))[:30],
                'P/E': pd.to_numeric(row.get('PERatio'), errors='coerce'),
                'EV/EBITDA': pd.to_numeric(row.get('EVToEBITDA'), errors='coerce'),
                'P/B': pd.to_numeric(row.get('PriceToBookRatio'), errors='coerce'),
                'ROE %': pd.to_numeric(row.get('ReturnOnEquityTTM'), errors='coerce') * 100,
                'Маржа %': pd.to_numeric(row.get('ProfitMargin'), errors='coerce') * 100,
                'Бета': pd.to_numeric(row.get('Beta'), errors='coerce'),
            })
        except Exception:
            pass

if records:
    valuation = pd.DataFrame(records).set_index('Тикер').sort_values('EV/EBITDA')
    print(f'Компаний в секторе {SECTOR}: {len(valuation)}')
    display(valuation.dropna(how='all').head(20))
else:
    print(f'Данные для сектора {SECTOR} не найдены')

In [ ]:
# Тепловая карта оценочных мультипликаторов
if records:
    import matplotlib.colors as mcolors

    metrics = ['P/E', 'EV/EBITDA', 'P/B', 'ROE %', 'Маржа %']
    heatmap_data = valuation[metrics].dropna(how='all').head(30)

    # Нормализуем каждый столбец для цветовой шкалы
    norm_data = heatmap_data.copy()
    for col in metrics:
        col_min = heatmap_data[col].min()
        col_max = heatmap_data[col].max()
        if col_max > col_min:
            norm_data[col] = (heatmap_data[col] - col_min) / (col_max - col_min)

    fig, ax = plt.subplots(figsize=(12, max(6, len(heatmap_data) * 0.4)))
    im = ax.imshow(norm_data.values, aspect='auto', cmap='RdYlGn_r')

    ax.set_xticks(range(len(metrics)))
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_yticks(range(len(heatmap_data)))
    ax.set_yticklabels(heatmap_data.index.tolist(), fontsize=9)

    for i in range(len(heatmap_data)):
        for j, col in enumerate(metrics):
            val = heatmap_data.iloc[i][col]
            if pd.notna(val):
                ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8)

    ax.set_title(f'Мультипликаторы компаний — сектор {SECTOR}', fontsize=13)
    plt.colorbar(im, ax=ax, label='Нормализованное значение')
    plt.tight_layout()
    plt.show()

## 10. Динамика EBITDA по всему сектору

In [ ]:
SECTOR = 'Technology'  # измените под себя
TOP_N = 10             # топ N компаний по последнему значению EBITDA

pivot = load_all_tickers_report('income_statement_annual', 'ebitda', sector_filter=SECTOR)

if not pivot.empty:
    # Топ N по последнему значению
    top_tickers = pivot.iloc[-1].nlargest(TOP_N).index.tolist()
    pivot_top = pivot[top_tickers].dropna(how='all')

    fig, ax = plt.subplots(figsize=(14, 7))
    for ticker in top_tickers:
        s = pivot_top[ticker].dropna()
        ax.plot(s.index, s.values / 1e9, marker='o', label=ticker, linewidth=2)

    ax.set_title(f'Топ-{TOP_N} компаний по EBITDA — сектор {SECTOR}', fontsize=14)
    ax.set_ylabel('млрд USD')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'Нет данных для сектора {SECTOR}')

## 11. Сводная таблица: Топ-20 компаний по рыночной капитализации

In [ ]:
all_overviews = []

for sector_dir in DATA_DIR.iterdir():
    if not sector_dir.is_dir() or sector_dir.name == '__pycache__':
        continue
    for ticker_dir in sector_dir.iterdir():
        ov_path = ticker_dir / 'overview.csv'
        if not ov_path.exists():
            continue
        try:
            ov = pd.read_csv(ov_path)
            if ov.empty:
                continue
            row = ov.iloc[0].to_dict()
            row['_sector'] = sector_dir.name
            all_overviews.append(row)
        except Exception:
            pass

if all_overviews:
    ov_df = pd.DataFrame(all_overviews)
    numeric_cols = ['MarketCapitalization', 'EBITDA', 'RevenueTTM',
                    'PERatio', 'EVToEBITDA', 'ReturnOnEquityTTM', 'ProfitMargin', 'Beta']
    for col in numeric_cols:
        if col in ov_df.columns:
            ov_df[col] = pd.to_numeric(ov_df[col], errors='coerce')

    top20 = ov_df.nlargest(20, 'MarketCapitalization')[[
        'Symbol', 'Name', '_sector', 'MarketCapitalization',
        'EBITDA', 'RevenueTTM', 'PERatio', 'EVToEBITDA',
        'ReturnOnEquityTTM', 'ProfitMargin'
    ]].rename(columns={
        '_sector': 'Сектор',
        'MarketCapitalization': 'Капит. (млрд)',
        'EBITDA': 'EBITDA (млрд)',
        'RevenueTTM': 'Выручка TTM (млрд)',
        'PERatio': 'P/E',
        'EVToEBITDA': 'EV/EBITDA',
        'ReturnOnEquityTTM': 'ROE',
        'ProfitMargin': 'Маржа'
    })

    for col in ['Капит. (млрд)', 'EBITDA (млрд)', 'Выручка TTM (млрд)']:
        top20[col] = (top20[col] / 1e9).round(1)

    top20.reset_index(drop=True, inplace=True)
    display(top20)
else:
    print('Данные overview не найдены. Запустите download_fundamentals.py')

## 12. Скаттер-плот: ROE vs P/E по секторам

In [ ]:
if all_overviews:
    plot_df = ov_df.dropna(subset=['ReturnOnEquityTTM', 'PERatio']).copy()
    plot_df = plot_df[
        (plot_df['PERatio'] > 0) & (plot_df['PERatio'] < 100) &
        (plot_df['ReturnOnEquityTTM'] > -1) & (plot_df['ReturnOnEquityTTM'] < 2)
    ]

    sectors = plot_df['_sector'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(sectors)))
    color_map = dict(zip(sectors, colors))

    fig, ax = plt.subplots(figsize=(14, 8))

    for sector, group in plot_df.groupby('_sector'):
        ax.scatter(
            group['PERatio'],
            group['ReturnOnEquityTTM'] * 100,
            label=sector,
            color=color_map[sector],
            alpha=0.7, s=60,
        )

    ax.set_xlabel('P/E', fontsize=12)
    ax.set_ylabel('ROE (%)', fontsize=12)
    ax.set_title('ROE vs P/E по секторам', fontsize=14)
    ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
    ax.axhline(0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()